# Palace Mesh Generation: IHP RF Devices

This notebook discovers RF components from `ihp.cells.rf_devices`, prints their ports, and generates Palace meshes for each component.

**Requirements:**
- IHP PDK from GitHub (`pip install git+https://github.com/gdsfactory/IHP.git`)
- gsim environment with Palace meshing dependencies

In [ ]:
from __future__ import annotations

from pathlib import Path

import ihp.cells.rf_devices as rf_devices
import pandas as pd
from ihp import PDK

from gsim.common.stack import get_stack
from gsim.palace import DrivenSim

PDK.activate()

In [ ]:
# Discover RF component factories defined in ihp.cells.rf_devices and registered in the active PDK.
rf_module_factories = {
    name
    for name, obj in vars(rf_devices).items()
    if callable(obj) and not name.startswith("_")
}

rf_keywords = (
    "wilkinson",
    "filter",
    "coupler",
    "divider",
    "directional",
    "hybrid",
    "branch",
    "transformer",
)

component_names = sorted(
    name
    for name in PDK.cells
    if name in rf_module_factories and any(k in name.lower() for k in rf_keywords)
)

print("Discovered RF components:")
for name in component_names:
    print(f"- {name}")

if not component_names:
    raise RuntimeError(
        "No RF device components were discovered in ihp.cells.rf_devices"
    )

Discovered RF components:
- branch_line_coupler
- coupled_line_bandpass_filter
- coupler_tline
- directional_coupler
- hairpin_coupled_line_bandpass_filter
- quarter_wave_transformer
- wilkinson_power_divider


In [ ]:
# Build each component and tabulate ports.
port_rows = []
components = {}

for name in component_names:
    c = PDK.cells[name]()
    components[name] = c
    port_rows.extend(
        {
            "component": name,
            "port": p.name,
            "orientation_deg": p.orientation,
            "width_um": p.width,
            "center_um": tuple(float(x) for x in p.center),
            "layer": str(p.layer),
        }
        for p in c.ports
    )

ports_df = (
    pd.DataFrame(port_rows).sort_values(["component", "port"]).reset_index(drop=True)
)
ports_df

,component,port,orientation_deg,width_um,center_um,layer
0,branch_line_coupler,e1,180.0,1.68,"(-100.0, 0.84)",TopMetal2drawing
1,branch_line_coupler,e2,0.0,1.68,"(3838.657, 0.84)",TopMetal2drawing
2,branch_line_coupler,e3,0.0,1.68,"(3838.657, -3733.505)",TopMetal2drawing
3,branch_line_coupler,e4,180.0,1.68,"(-100.0, -3733.505)",TopMetal2drawing
4,coupled_line_bandpass_filter,e1,180.0,1.68,"(0.0, 0.0)",TopMetal2drawing
5,coupled_line_bandpass_filter,e2,0.0,1.68,"(15037.564, 25.204)",TopMetal2drawing
6,coupler_tline,e1,180.0,1.68,"(0.0, 13.442)",TopMetal2drawing
7,coupler_tline,e2,0.0,1.68,"(10.0, 13.442)",TopMetal2drawing
8,coupler_tline,e3,0.0,1.68,"(10.0, -13.442)",TopMetal2drawing
9,coupler_tline,e4,180.0,1.68,"(0.0, -13.442)",TopMetal2drawing


In [4]:
# Generate a mesh per component under ./palace-sim-ihp-rf/<component_name>.
stack = get_stack()
base_dir = Path("./palace-sim-ihp-rf")
base_dir.mkdir(parents=True, exist_ok=True)

mesh_rows = []

for name, c in components.items():
    out_dir = base_dir / name
    out_dir.mkdir(parents=True, exist_ok=True)

    try:
        sim = DrivenSim()
        sim.set_output_dir(out_dir)
        sim.set_geometry(c)
        sim.set_stack(stack)

        # Add all component ports as via ports so port geometries are meshed.
        for port in c.ports:
            sim.add_port(
                port.name,
                from_layer="metal3",
                to_layer="topmetal2",
                geometry="via",
            )

        sim.set_driven(fmin=1e9, fmax=110e9, num_points=21)
        result = sim.mesh(preset="default")

        mesh_rows.append(
            {
                "component": name,
                "status": "ok",
                "mesh_path": str(result.mesh_path),
            }
        )
    except Exception as e:
        mesh_rows.append(
            {
                "component": name,
                "status": "error",
                "mesh_path": "",
                "error": str(e),
            }
        )

mesh_df = (
    pd.DataFrame(mesh_rows).sort_values(["status", "component"]).reset_index(drop=True)
)
mesh_df

Small conductor feature detected (1.680 um) may be under-resolved by refined_mesh_size=5.000 um. Pass auto_size=True to scale the mesh down.
Small conductor feature detected (1.634 um) may be under-resolved by refined_mesh_size=5.000 um. Pass auto_size=True to scale the mesh down.
Small conductor feature detected (1.680 um) may be under-resolved by refined_mesh_size=5.000 um. Pass auto_size=True to scale the mesh down.
Small conductor feature detected (0.536 um) may be under-resolved by refined_mesh_size=5.000 um. Pass auto_size=True to scale the mesh down.
Small conductor feature detected (1.078 um) may be under-resolved by refined_mesh_size=5.000 um. Pass auto_size=True to scale the mesh down.
Small conductor feature detected (0.200 um) may be under-resolved by refined_mesh_size=5.000 um. Pass auto_size=True to scale the mesh down.
Small conductor feature detected (0.200 um) may be under-resolved by refined_mesh_size=5.000 um. Pass auto_size=True to scale the mesh down.


,component,status,mesh_path
0,branch_line_coupler,ok,palace-sim-ihp-rf/branch_line_coupler/palace.msh
1,coupled_line_bandpass_filter,ok,palace-sim-ihp-rf/coupled_line_bandpass_filter...
2,coupler_tline,ok,palace-sim-ihp-rf/coupler_tline/palace.msh
3,directional_coupler,ok,palace-sim-ihp-rf/directional_coupler/palace.msh
4,hairpin_coupled_line_bandpass_filter,ok,palace-sim-ihp-rf/hairpin_coupled_line_bandpas...
5,quarter_wave_transformer,ok,palace-sim-ihp-rf/quarter_wave_transformer/pal...
6,wilkinson_power_divider,ok,palace-sim-ihp-rf/wilkinson_power_divider/pala...


In [ ]:
from IPython.display import Markdown, display

from gsim.viz import plot_mesh

# Prefer an inline interactive backend in notebooks when available.
try:
    import pyvista as pv

    pv.set_jupyter_backend("trame")
except Exception:
    pass

transparent_groups = ["sin__None", "si__None", "Absorbing_boundary"]

shown = 0
for row in mesh_df.to_dict(orient="records"):
    if row.get("status") != "ok":
        continue

    component = str(row["component"])
    msh_path = Path(row["mesh_path"])

    if not msh_path.exists():
        continue

    shown += 1
    display(Markdown(f"### {component}"))
    plot_mesh(
        msh_path,
        interactive=True,
        style="solid",
        transparent_groups=transparent_groups,
    )

if shown == 0:
    raise RuntimeError(
        "No successful meshes found to display. Run the meshing cell first."
    )

### branch_line_coupler

Widget(value='<iframe src="http://localhost:41703/index.html?ui=P_0x73e10dd7d8b0_0&reconnect=auto" class="pyvi…

### coupled_line_bandpass_filter

Widget(value='<iframe src="http://localhost:41703/index.html?ui=P_0x73e38165c620_1&reconnect=auto" class="pyvi…

### coupler_tline

...

Widget(value='<iframe src="http://localhost:41703/index.html?ui=P_0x73e1b07c5340_4&reconnect=auto" class="pyvi…

### quarter_wave_transformer

Widget(value='<iframe src="http://localhost:41703/index.html?ui=P_0x73e20c7054f0_5&reconnect=auto" class="pyvi…

### wilkinson_power_divider

Widget(value='<iframe src="http://localhost:41703/index.html?ui=P_0x73e2381c7aa0_6&reconnect=auto" class="pyvi…